# Reconstruction Example Using lftomo

Environment Requirements:

See x

In [ ]:
from multiprocessing import shared_memory, Pool
from multiprocessing.managers import SharedMemoryManager
import os
from pathlib import Path
import sys

# Image Display
import cv2
from ipywidgets import Image
# Correct ordering if input not zero-padded
from natsort import natsorted
# Used for everything
import numpy as np
# Used to get processor data for setting multiprocessing.
import psutil
# Default file format is tif
import tifffile as tf
# Used for Recon
import tomopy

sys.path.insert(0, os.path.abspath('..'))

# Flexible logging to screen and file, in a parseable fixed-width format.
from lftomo.util.log import log
# Windowing to select only parts of an image at a time.
from lftomo.util.window import DataWindowSet, ProjOrder, ReconOrder, SinoOrder, get_tilt_pad, CTFov, ImgShape
# Handling flatfield backgrounds.
from lftomo.prep.flat import FLAT, weighted_normalize, calculate_shift
# Shared Memory Handling
from lftomo.util.mem import SharedNP
# Reading and Writing files distributed
from lftomo.util.io import distribute_read, write_slice, np_convert, write_centers
# Stripe removal
from lftomo.prep.stripe import remove_sort_vo, remove_large_vo, remove_dead_vo

## Setup

### Reconstruction Parameters

Starting with basics (input path, window size, output location)

In [ ]:
log.set_logdir(Path("..", "data", "logs"))
log.header()
# smm = SharedMemoryManager()
# smm.start()

In [ ]:

window_size = psutil.cpu_count()
input_path = Path("..", "data", "input", "Sam19")
flat_path = Path("..", "data", "input", "Sam19_Flats")
output_base = Path("..", "data", "output")

# Field of View Angles.
fov = CTFov(Theta=None, Degrees=180, Count=1)

internal_dtype = np.float32		# Data type to use internally. 
cf_count = 8 					# Number of centerfind sections to generate.
proc_count = 12					# Number of simulataneous processes.
center_count = 20				# Number of centers to generate.

In [ ]:
data_set = DataWindowSet.from_folder(input_path, ProjOrder, window_size, fov)
input_files = natsorted(Path(input_path).glob("**/*.tif*"))

### Flat Field Generation

In [ ]:
flat_mem = SharedNP(f'flats_ipynb', internal_dtype, data_set.data_shape._replace(Theta=len(FLAT))).create()

with flat_mem as flat_set:
	for flat in list(FLAT):
		flat_set[flat.index, :, :] = tf.imread(flat_path.joinpath(f"{flat}_median.tiff")).astype(internal_dtype)
	
	# Precalculation of pregain and postgain with darks.
	flat_set[FLAT.PREGAIN.index] -= flat_set[FLAT.PREDARK.index]
	flat_set[FLAT.POSTGAIN.index] -= flat_set[FLAT.POSTDARK.index]


In [ ]:
with flat_mem as flat_set:
	flat_intermediate = output_base.joinpath("flat_intermediate")
	flat_intermediate.mkdir(parents=True, exist_ok=True)
	tf.imwrite(Path(flat_intermediate, "pregain.tiff"), flat_set[FLAT.PREGAIN.index])
	tf.imwrite(Path(flat_intermediate, "postgain.tiff"), flat_set[FLAT.POSTGAIN.index])
	tf.imwrite(Path(flat_intermediate, "predark.tiff"), flat_set[FLAT.PREDARK.index])
	tf.imwrite(Path(flat_intermediate, "postdark.tiff"), flat_set[FLAT.POSTDARK.index])

	if False:
		# Encode into in-memory PNG for Jupyter
		# _, pregain = cv2.imencode('.png', flat_set[FLAT.PREGAIN.index])
		# _, postgain = cv2.imencode('.png', flat_set[FLAT.POSTGAIN.index])

		# This 'works' but requires leveling
		Image(value=pregain, width=data_set.data_shape.X, height=data_set.data_shape.Y)

### Angle Calculation

In [ ]:
# No angle dropping so skipping for now.

## Center Finding

Steps:

1. Read Slice(s)
2. Data Cleanup
3. Center Calculation
4. Angle Calculation
5. Reconstruct Slices
6. Normalize Slices


In [ ]:
center_shape = SinoOrder(**data_set.data_shape._asdict())._replace(Y=cf_count)
cf_read_mem = SharedNP(f"cf_read_ipynb", data_set.dtype, center_shape).create()

cf_dataset = data_set.at_interval(cf_count)
centers = {}


for i, cf_data in enumerate(cf_dataset):
	log.write("Center Finding", f"{cf_data}, {cf_read_mem.buffer_address}")
	distribute_read(cf_read_mem[i], cf_data, input_files, sino_order=True)
	with cf_read_mem as cf_slice:
		log.write("CF Test", f"{cf_slice.shape}")
		centers[cf_data.start] = tomopy.find_center_vo(cf_slice, i)
		

log.write("Center Finds", f"Vo CFs: {centers}")

In [ ]:
cf_mem = SharedNP(f"cf_ipynp", internal_dtype, SinoOrder(**data_set.data_shape._asdict())._replace(Y=cf_count)).create()
cf_win = cf_dataset.full_window()

with Pool(proc_count) as pool:
	pool.starmap(weighted_normalize, [(cf_mem, flat_mem, cf_read_mem, cf_win, i, True) for i in range(cf_mem.shape.Theta)])
	log.write("Normalize", f"{cf_win.internal.Y()}|{cf_win.Y()}")
	pool.starmap(remove_dead_vo, [(cf_mem, i) for i in cf_win.internal.y_range])
	log.write("Stripe Removal (D,L)", f"{cf_win.internal.Y()}|{cf_win.Y()}")
# 	pool.starmap(remove_large_vo, [(cf_mem, i) for i in cf_win.internal.y_range])
#	log.write("Stripe Removal (L)", f"{cf_win.internal.Y()}|{cf_win.Y()}")

remove_sort_vo(cf_win, cf_mem)
log.write("Stripe Removal (s)", f"{cf_win.internal.Y()}|{cf_win.Y()}")


In [ ]:
with cf_mem as cf_slice:
	for i, cf_data in enumerate(cf_dataset):
		log.write("CF Test", f"{cf_slice.shape}")
		centers[cf_data.start] = tomopy.find_center_vo(cf_slice, i)
log.write("Center Finds", f"Vo CFs: {centers}")

base_center = tomopy.find_center_pc(tf.imread(input_files[0]), tf.imread(input_files[1]))
log.write("Center Find", f"Tomopy Center {base_center}")
base_center = 1036.0

In [ ]:
entropy_centers = {}
base_entropy_centers = {}
for i, cf_data in enumerate(cf_dataset):
	log.write("Center Writing", f"{len(data_set.angles)}_{cf_mem.shape})")
	center_path = output_base.joinpath("cf_data", f"s{str(cf_data.start).zfill(5)}")
	center_path.mkdir(parents=True, exist_ok=True)
	write_centers(cf_mem[i], center_path, np.linspace(base_center - 5.0, base_center + 5.0, center_count), data_set.angles, calc_entropy=False)

	center_path = output_base.joinpath("cf_data", f"noNorm_s{str(cf_data.start).zfill(5)}")
	center_path.mkdir(parents=True, exist_ok=True)
	write_centers(cf_read_mem[i], center_path, np.linspace(base_center - 5.0, base_center + 5.0, center_count), data_set.angles, calc_entropy=False)

cf_read_mem.close()
cf_read_mem.unlink()

# log.write("Center Find", f"Entropy Centers {entropy_centers}")
# log.write("Center Find", f"Base Entropy Centers {base_entropy_centers}")


In [ ]:
cf_mem.close()
cf_mem.unlink()

## Reconstruction

Loop, looking at small windows each time.

1. Read Data
2. Clean Data (Pre-Rotation Steps)
3. Rotate Data
4. Clean Data (Post-Rotation Steps)
5. Reconstruct Slices
6. Normalize Slices

In [ ]:
with (SharedNP(f"sino_ipynb", internal_dtype, SinoOrder(**data_set.window_shape._asdict())) as sino_mem,
	SharedNP(f"workread_ipynb", data_set.dtype, SinoOrder(**data_set.window_shape._asdict())) as workmem_read,
	SharedNP(f"recon_ipynb", internal_dtype, ReconOrder(Y=data_set.window_size, Z=data_set.window_shape.X, X=data_set.window_shape.X)) as recon_mem):

	for j, dw in enumerate(data_set):
		log.write("Reading Data", f"{data_set.window_shape} : {dw} : {workmem_read.shape} : {sino_mem.shape}")
		distribute_read(workmem_read[dw.internal.Y()], dw, input_files, sino_order=True)
		log.write("Data Read")

		with Pool(proc_count) as pool:
			log.write("Normalize Prep", f"{dw.internal.Y()}|{dw.Y()}")
			pool.starmap(weighted_normalize, [(sino_mem, flat_mem, workmem_read, dw, i, True) for i in range(sino_mem.shape.Theta)])
			log.write("Flat Field Adjustment", f"{dw.internal.Y()}|{dw.Y()}")
			pool.starmap(remove_dead_vo, [(sino_mem, i) for i in dw.internal.y_range])
			log.write("Stripe Removal (D,L)", f"{dw.internal.Y()}|{dw.Y()}")

		remove_sort_vo(dw, sino_mem)
		log.write("Stripe Removal (S)", f"{dw.internal.Y()}|{dw.Y()}")

		with sino_mem as input_data, recon_mem as output_data:
			output_data[:, :, :] = tomopy.recon(input_data[:, :, :], data_set.angles, sinogram_order=True, center=base_center, algorithm="gridrec", ncore=proc_count)	

		with Pool(proc_count) as pool:
			output_base.joinpath("recon").mkdir(exist_ok=True, parents=True)
			log.write("Writing Recon", f"{dw.Y()}")

			pool.starmap(write_slice, [(recon_mem[0:data_set.window_size], output_base.joinpath("recon", f"{str(i).zfill(5)}.tif"), i - dw.Y().start)
											for i in dw.y_range])


In [ ]:
flat_mem.close()
flat_mem.unlink()

In [ ]:
sino_mem.close()
sino_mem.unlink()
workmem_read.close()
workmem_read.unlink()
recon_mem.close()
recon_mem.unlink()